# ⚡ Day 03a: I/O Systems — Polling, Interrupts, DMA & RAID

---

## 🎯 What You'll Master Today
- I/O handling: polling vs interrupts vs DMA
- Device drivers and I/O subsystem layers
- RAID levels (0, 1, 5, 6, 10)
- Buffering and spooling

---

```
╔══════════════════════════════════════════════════════════════╗
║              I/O HANDLING METHODS                            ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Polling:     CPU repeatedly checks "data ready?"            ║
║               Simple but wastes CPU cycles (busy waiting)    ║
║                                                              ║
║  Interrupts:  Device signals CPU when ready                  ║
║               CPU free between checks, overhead per transfer ║
║                                                              ║
║  DMA:         DMA controller handles bulk transfer           ║
║               CPU only involved at start/end (1 interrupt)   ║
║               Best for large data transfers (disk, network)  ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. I/O Method Comparison Simulation
# ============================================

import time

class IOSimulator:
    """Compare polling vs interrupt vs DMA overhead."""
    
    def __init__(self, data_blocks: int, cpu_cycle_ns: int = 1):
        self.data_blocks = data_blocks
        self.cpu_cycle = cpu_cycle_ns  # nanoseconds
    
    def polling(self) -> dict:
        """CPU busy-waits, checking status register each cycle."""
        checks_per_block = 100  # avg polls before data ready
        total_polls = self.data_blocks * checks_per_block
        cpu_wasted = total_polls * self.cpu_cycle
        return {
            "method": "Polling",
            "cpu_busy_cycles": total_polls,
            "interrupts": 0,
            "cpu_free": False,
            "best_for": "Fast devices, short waits"
        }
    
    def interrupt_driven(self) -> dict:
        """Device interrupts CPU when each block is ready."""
        interrupt_overhead = 500  # cycles per interrupt (context switch)
        total_interrupts = self.data_blocks
        cpu_overhead = total_interrupts * interrupt_overhead
        return {
            "method": "Interrupt-Driven",
            "cpu_busy_cycles": cpu_overhead,
            "interrupts": total_interrupts,
            "cpu_free": True,
            "best_for": "Moderate I/O, CPU can do other work"
        }
    
    def dma(self) -> dict:
        """DMA controller handles entire transfer, 1 interrupt at end."""
        setup_cycles = 1000     # DMA setup cost
        interrupt_overhead = 500
        return {
            "method": "DMA",
            "cpu_busy_cycles": setup_cycles + interrupt_overhead,
            "interrupts": 1,
            "cpu_free": True,
            "best_for": "Large bulk transfers (disk, network)"
        }


# Transfer 1000 blocks
sim = IOSimulator(data_blocks=1000)

print("  I/O Method Comparison (1000 blocks)")
print("  " + "═" * 55)

for method in [sim.polling, sim.interrupt_driven, sim.dma]:
    result = method()
    cpu = result['cpu_busy_cycles']
    bar = '█' * min(int(cpu / 1000), 30)
    print(f"\n  {result['method']}:")
    print(f"    CPU Cycles Used: {cpu:>8} {bar}")
    print(f"    Interrupts:      {result['interrupts']}")
    print(f"    CPU Free:        {'Yes ✅' if result['cpu_free'] else 'No ❌'}")
    print(f"    Best For:        {result['best_for']}")

In [ ]:
# ============================================
# 2. RAID Levels — Comparison
# ============================================

class RAIDCalculator:
    """Calculate RAID usable capacity, fault tolerance, performance."""
    
    @staticmethod
    def calculate(level: str, num_disks: int, disk_size_gb: int) -> dict:
        configs = {
            "RAID 0": {
                "name": "Striping",
                "usable": num_disks * disk_size_gb,
                "overhead": 0,
                "fault_tolerance": 0,
                "min_disks": 2,
                "read_speed": f"{num_disks}x",
                "write_speed": f"{num_disks}x",
            },
            "RAID 1": {
                "name": "Mirroring",
                "usable": num_disks // 2 * disk_size_gb,
                "overhead": 50,
                "fault_tolerance": num_disks // 2,
                "min_disks": 2,
                "read_speed": f"{num_disks}x",
                "write_speed": "1x",
            },
            "RAID 5": {
                "name": "Striping + Distributed Parity",
                "usable": (num_disks - 1) * disk_size_gb,
                "overhead": round(100 / num_disks),
                "fault_tolerance": 1,
                "min_disks": 3,
                "read_speed": f"{num_disks - 1}x",
                "write_speed": "slower (parity calc)",
            },
            "RAID 6": {
                "name": "Striping + Double Parity",
                "usable": (num_disks - 2) * disk_size_gb,
                "overhead": round(200 / num_disks),
                "fault_tolerance": 2,
                "min_disks": 4,
                "read_speed": f"{num_disks - 2}x",
                "write_speed": "slowest (2x parity)",
            },
            "RAID 10": {
                "name": "Mirror + Stripe",
                "usable": num_disks // 2 * disk_size_gb,
                "overhead": 50,
                "fault_tolerance": num_disks // 2,
                "min_disks": 4,
                "read_speed": f"{num_disks}x",
                "write_speed": f"{num_disks // 2}x",
            },
        }
        return configs.get(level, {})


# Compare RAID levels with 6 × 1TB disks
num_disks = 6
disk_size = 1  # TB

print(f"  RAID Comparison: {num_disks} × {disk_size}TB disks")
print(f"  {'═' * 60}")

for level in ["RAID 0", "RAID 1", "RAID 5", "RAID 6", "RAID 10"]:
    r = RAIDCalculator.calculate(level, num_disks, disk_size)
    print(f"\n  {level} ({r['name']}):")
    print(f"    Usable:    {r['usable']}TB / {num_disks * disk_size}TB ({100 - r['overhead']}%)")
    print(f"    Tolerate:  {r['fault_tolerance']} disk failure(s)")
    print(f"    Read:      {r['read_speed']} | Write: {r['write_speed']}")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Polling vs interrupts? | Polling: CPU busy-waits. Interrupts: CPU free, overhead per transfer |
| 2 | When is DMA used? | Large bulk transfers (disk reads, network packets). CPU only at start/end |
| 3 | RAID 5 vs RAID 10? | RAID 5: space efficient (1 disk parity). RAID 10: faster, tolerates more failures |
| 4 | What is spooling? | Queue for slow I/O devices (printer). Jobs wait in buffer, processed sequentially |
| 5 | Buffering vs caching? | Buffer: hold data in transit. Cache: keep copy for faster re-access |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Polling: simple, wastes CPU. Good for fast devices    │
## │  • Interrupts: efficient, overhead per event             │
## │  • DMA: best for bulk I/O, 1 interrupt per transfer      │
## │  • RAID 0: speed, no safety. RAID 1: mirror. RAID 5/6: parity │
## │  • RAID 10 = production standard (speed + redundancy)    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Linux Process Commands** — ps, top, kill, cron, systemd